# SkinCoach — Обучение модели v3.1
**Датасеты:** HAM10000 + Fitzpatrick17k (опционально)  
**Модель:** EfficientNet-B3 (N классов — определяется анализом данных)  
**Цель:** val_acc > 80%

## Как запустить на Kaggle:
1. Создать новый ноутбук на [kaggle.com/code](https://www.kaggle.com/code)
2. Settings → Accelerator → **GPU T4 x2**
3. Добавить датасет: **Skin cancer: HAM10000** (от surajghuwalewala)
4. Установить секреты в Kaggle → Add-ons → Secrets:
   - `HF_TOKEN` = ваш HuggingFace write token
   - `GITHUB_TOKEN` = ваш GitHub Personal Access Token (если репо приватный)
5. Нажать **Run All**

In [ ]:
# ── Шаг 0: Проверка GPU ──────────────────────────────────────────────────────
import subprocess
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(result.stdout if result.returncode == 0 else '⚠️ GPU не найден')

In [ ]:
# ── Шаг 1: Установка зависимостей ────────────────────────────────────────────
!pip install -q huggingface_hub torch torchvision pillow

In [ ]:
# ── Шаг 2: Клонируем репозиторий SkinCoach ───────────────────────────────────
import os

# Пробуем взять GitHub токен из Kaggle Secrets (для приватного репо)
GITHUB_TOKEN = ''
try:
    from kaggle_secrets import UserSecretsClient
    secrets = UserSecretsClient()
    GITHUB_TOKEN = secrets.get_secret('GITHUB_TOKEN')
    print('GitHub токен загружен из Kaggle Secrets')
except:
    print('GITHUB_TOKEN не найден в Secrets — используем публичный доступ')

if GITHUB_TOKEN:
    REPO_URL = f'https://{GITHUB_TOKEN}@github.com/lipindanil02-eng/skincoach08.03.git'
else:
    REPO_URL = 'https://github.com/lipindanil02-eng/skincoach08.03.git'

if not os.path.exists('/kaggle/working/skincoach'):
    os.system(f'git clone {REPO_URL} /kaggle/working/skincoach')
else:
    print('Репозиторий уже скачан')

os.chdir('/kaggle/working/skincoach')
print('Рабочая директория:', os.getcwd())
print('Файлы:', os.listdir('.'))

In [ ]:
# ── Шаг 3: Автоматическое определение путей к датасетам ──────────────────────
import os

# Автоматически находим HAM10000 в /kaggle/input/
HAM_DIR = None
for root, dirs, files in os.walk('/kaggle/input'):
    if 'GroundTruth.csv' in files or 'HAM10000_metadata.csv' in files:
        HAM_DIR = root
        break
    if 'images' in dirs and any(f.endswith('.csv') for f in files):
        HAM_DIR = root
        break

if not HAM_DIR:
    # Пробуем стандартные пути
    for candidate in [
        '/kaggle/input/ham10000',
        '/kaggle/input/skin-cancer-ham10000',
    ]:
        if os.path.exists(candidate):
            HAM_DIR = candidate
            break
    # Если ничего не нашли — ищем рекурсивно любой CSV с изображениями
    if not HAM_DIR:
        for root, dirs, files in os.walk('/kaggle/input'):
            csvs = [f for f in files if f.endswith('.csv')]
            if csvs and 'images' in dirs:
                HAM_DIR = root
                break

# Fitzpatrick17k — скачивается в шаге 4
FITZ_DIR = '/kaggle/working/fitzpatrick17k'

# Папка для итогового датасета
OUT_DIR = '/kaggle/working/dataset'

# Минимальное количество фото для отдельного класса
MIN_COUNT = 50

print(f'HAM_DIR = {HAM_DIR}')
print(f'HAM10000 найден: {HAM_DIR is not None and os.path.exists(HAM_DIR)}')
if HAM_DIR and os.path.exists(HAM_DIR):
    print(f'Содержимое: {os.listdir(HAM_DIR)}')

In [ ]:
# ── Шаг 4: Скачиваем Fitzpatrick17k (опционально) ────────────────────────────
# Fitzpatrick17k требует запрос доступа у авторов
# Если нет доступа — обучение пойдёт только на HAM10000 (это ОК для MVP)

if not os.path.exists(FITZ_DIR):
    print('Клонируем метаданные Fitzpatrick17k...')
    os.system(f'git clone https://github.com/mattgroh/fitzpatrick17k.git {FITZ_DIR}')
    if os.path.exists(f'{FITZ_DIR}/fitzpatrick17k.csv'):
        print('Fitzpatrick17k CSV найден')
        print('Для полного обучения нужно скачать изображения с сайта авторов')
    else:
        print('Fitzpatrick17k CSV не найден — пропускаем')
else:
    print('Fitzpatrick17k уже скачан')

print(f'\nСтатус: Fitzpatrick17k = {os.path.exists(FITZ_DIR)}')

In [ ]:
# ── Шаг 5: Анализ датасетов ──────────────────────────────────────────────────
import os

cmd = 'python analyze_datasets.py'
if HAM_DIR and os.path.exists(HAM_DIR):
    cmd += f' --ham {HAM_DIR}'
if os.path.exists(f'{FITZ_DIR}/fitzpatrick17k.csv'):
    cmd += f' --fitz {FITZ_DIR}'
cmd += f' --threshold {MIN_COUNT}'

print(f'Запускаю: {cmd}')
os.system(cmd)

In [ ]:
# ── Шаг 6: Подготовка датасета ────────────────────────────────────────────────
import os

cmd = f'python prepare_dataset.py --out {OUT_DIR} --min_count {MIN_COUNT}'
if HAM_DIR and os.path.exists(HAM_DIR):
    cmd += f' --ham {HAM_DIR}'
if os.path.exists(f'{FITZ_DIR}/fitzpatrick17k.csv'):
    cmd += f' --fitz {FITZ_DIR}'

print(f'Запускаю: {cmd}')
os.system(cmd)

In [ ]:
# ── Шаг 7: Статистика датасета ────────────────────────────────────────────────
import os

print('📊 Итоговый датасет:\n')
total_train, total_val = 0, 0
for cls in sorted(os.listdir(os.path.join(OUT_DIR, 'train'))):
    n_train = len(os.listdir(os.path.join(OUT_DIR, 'train', cls)))
    n_val   = len(os.listdir(os.path.join(OUT_DIR, 'val',   cls))) if os.path.exists(os.path.join(OUT_DIR, 'val', cls)) else 0
    total_train += n_train
    total_val   += n_val
    status = '✅' if n_train >= 100 else ('⚠️ ' if n_train > 0 else '❌')
    print(f'  {status} {cls:25s} train={n_train:5d}  val={n_val:4d}')

print(f'\n  Итого: train={total_train}, val={total_val}')

In [ ]:
# ── Шаг 8: ОБУЧЕНИЕ ───────────────────────────────────────────────────────────
import os

# HuggingFace токен
HF_TOKEN = ''
try:
    from kaggle_secrets import UserSecretsClient
    secrets = UserSecretsClient()
    HF_TOKEN = secrets.get_secret('HF_TOKEN')
    print('HuggingFace токен загружен')
except:
    print('HF_TOKEN не найден — модель не будет загружена на HuggingFace')
    print('   Добавьте токен: Kaggle → Add-ons → Secrets')

HF_REPO = 'danyil163/SCINCOACH'

hf_args = ''
if HF_TOKEN:
    hf_args = f'--hf_repo {HF_REPO} --hf_token {HF_TOKEN}'

cmd = f'python train.py --data {OUT_DIR} --epochs 40 --batch 32 --lr 0.0001 --out /kaggle/working/best_model.pth --workers 4 {hf_args}'
print(f'Запускаю: python train.py ...')
os.system(cmd)

In [ ]:
# ── Шаг 9: Проверка модели ────────────────────────────────────────────────────
import json

with open('class_map.json') as f:
    cm = json.load(f)
print('class_map.json:', json.dumps(cm, indent=2, ensure_ascii=False))

import os
size_mb = os.path.getsize('/kaggle/working/best_model.pth') / 1024 / 1024
print(f'\nРазмер модели: {size_mb:.1f} MB')

## После обучения

1. Скачайте `best_model.pth` и `class_map.json` (Output → кнопка Download)
2. Загрузите их на HuggingFace: `danyil163/SCINCOACH` (скрипт делает это автоматически если есть HF_TOKEN)
3. Railway автоматически подхватит новую модель при следующем cold start

**Целевые метрики:**
- val_acc > 80% — хорошо
- val_acc > 85% — отлично  
- val_acc > 70% для редких классов (vitiligo, rosacea) — приемлемо
